# Machine translation using the encoder-decoder model and RNNs

>That in his sund ride at whe thesess for her <br>
The sorter ullother in the that lakn and <br>
That in his sund ride at whe thesess for her <br>
Of which thou ars soon so to dis ande sarestes, <br>
That in his sund ride at whe thesess for her <br>
In her heed st ond wonde began and said, <br>
The sorter ullother in the that lakn and <br>
He was a thuse of lond and oo hor seist the cruptest the center. <br>
That in his sund ride at whe thesess for her <br>
That in his sund ride at whe thesess for her <br>
Geoffrey Chaucer, <i>Canterbury Tales</i>, lines 1-10, translated by the best model trained below

The goal of this lab activity is to understand how an encoder-decoder RNN is set up
and to observe widely-used libraries for implementing these models.
The premise and code is adapted from an article and accompanying codebase by Ravindra Kompella. See 
[the github repository](https://github.com/kmsravindra/ML-AI-experiments/blob/master/AI/Neural%20Machine%20Translation/Neural%20machine%20translation%20-%20Encoder-Decoder%20seq2seq%20model.ipynb)

The original article built an English-to-French translator, using a parallel corpus of English and French sentences. In our case, we'll begin with a simple translation task: translating from the language of decimal numbers to the language of Roman numerals,
that is

## 0. Coming up

In lieu of a "coming-up" slide, take note of the following things. (And this is it for the semester---nothing will be added.)

- Finish author/stylometry assignment (Fri, Dec 12)
- Do word2vec assignment (Fri, Dec 5)
- Read and annotate J&M Chapter 7, part 1 (Monday, Dec 8—class time)
- Read and respond to J&M Chapter 7, part 2 (Wednesday, Dec 10—class time)



## 1. RNNs, LSTMs, and encoder-decoder in Keras

You may recall from an earlier lab that `tensorflow` is a library for neural networks, including deep learning, and
`keras` is a subpackage that provides a useful interface to `tensorflow`. These imports grab what we need from keras and other packages. (You will get a bunch of warnings about cuda drivers being unavailable. This may affect performance, but the libraries should still work.)

In [1]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import Input, LSTM, Dense
import numpy as np
from random import randint, shuffle
from tensorflow.keras.utils import to_categorical

2025-12-05 13:36:32.693999: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-05 13:36:32.694337: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-05 13:36:32.696046: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-05 13:36:32.701244: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-05 13:36:32.710686: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

To get an idea of the entire set up being used, consider this diagram (which comes from Kompella's article), which
you can compare with figures we've seen from Jurafsky and Marin.


![encoder-decoder training network architecture for neural machine translation](https://cs.wheaton.edu/~tvandrun/cs384/kompella-encoder-decoder.png)

Study this diagram with your partner and get the general idea. You should then refer back to it as you see the part of it realized in the code below. A few things to take note of initially

- The light blue boxes in the decoder and light green boxes in the decoder are LSTMs, Accordingly, they output both a context vector and a hidden-state vector.
- This model tokenizes its input by character, not by word-token. The character embeddings, then capture some aspect of a characters "meaning", which must somehow reflect that meaning of all the words that character appears in.
- The *thought vector* (a term that I don't think Jurafsky and Martin use) is what is passed from the encoder to the decoder, containing all the information about the source sentence being translated; it is, apparently, the concatenation of the last context ($c_\mbox{final}$) and hidden state ($h_\mbox{final}$) of the encoder, which then becomes the initial context and hidden state for the decoder.
- The source sentence has a definite length, since it's the input (not a *fixed* length, though---different inputs have different lengths). The length of the output depends on how many characters are needed to make the output string. This means that the decoder somehow needs to know when to stop. This is done using `\n` as the terminal symbol. If the decoder ever produces a `\n`, then it stops.
- Since each decoder step takes the output of the previous step as its input, there needs to be some input for the first step. This example uses `\n` as the input to the initial decoder step. It's a "seed value" of sorts.
- Because of the previous two points, for each training sample, the output string begins with a `\t` and ends with a `\n`.

## 2. Translating to Roman numerals

Now let's try our first translation task. 

### 2.1 Getting the data
The file `rn.txt` contains every natural number less than 4000 as a decimal and as a 
Roman numeral. Each line of the file contains a data point. We put them in random order

In [2]:
lines = open('/homes/tvandrun/Public/cs384/rn.txt', encoding='utf-8').read().split('\n')
lines = lines[:-1]
shuffle(lines)

What do these look like?

In [3]:
lines[:10]

['662\tDCLXII',
 '3775\tMMMDCCLXXV',
 '1813\tMDCCCXIII',
 '545\tDXLV',
 '2632\tMMDCXXXII',
 '3453\tMMMCDLIII',
 '1647\tMDCXLVII',
 '3820\tMMMDCCCXX',
 '2894\tMMDCCCXCIV',
 '121\tCXXI']

We now split these lines into the decimal and Roman numeral values in parallel lists `decimal_samples` and `rn_samples`. Note that we pad the Roman numeral samples with a leading tab and trailing newline (actually *two* trailing newlines, for reasons we'll explain later). 

In [4]:
decimal_samples = []
rn_samples = []

for line in lines:
    decimal_line, rn_line = line.split('\t')
    rn_line = '\t' + rn_line + '\n\n'
    decimal_samples.append(decimal_line)
    rn_samples.append(rn_line)

nb_samples = len(decimal_samples)
assert nb_samples == len(rn_samples)

In [5]:
training_samples = 3000  # You could change this
decimal_train = decimal_samples[:training_samples]
decimal_test = decimal_samples[training_samples:]
rn_train = rn_samples[:training_samples]
rn_test = rn_samples[training_samples:]

See what these look like.

In [6]:
decimal_train[0]

'662'

In [7]:
rn_train[0]

'\tDCLXII\n\n'

### 2.2 Vectorizing the data for use in the model

Eventually we'll represent each character in either language as an embedding. To retrieve an embedding, each character needs to be turned into a one-hot vector. To assign one-hot vectors to characters, each character needs a unique index to identify it. The characters in the decimal number language are already numbers, so they can be their own index (`0` can be character number 0, `1` can be character number 1, etc). But for characters in the language of Roman numerals, we'll need to assign their index arbitrarily. So that we can convert between characters and indices, we have a dictionary for looking up a character's index and a list for looking up an index's character.

In [8]:
decimal_chars = set('0123456789')
rn_chars = set('MDLCXVI\t\n')
rn_index_to_char = list(rn_chars)
rn_char_to_index = {rn_index_to_char[i]:i for i in range(len(rn_index_to_char))}


Observe the result:

In [9]:
rn_index_to_char

['X', '\n', 'D', 'I', 'L', 'C', '\t', 'M', 'V']

In [10]:
rn_char_to_index

{'X': 0, '\n': 1, 'D': 2, 'I': 3, 'L': 4, 'C': 5, '\t': 6, 'M': 7, 'V': 8}

To vectorize this data, we make each data point to be of the same size. Each source and target string will be turned into a sequence of one-hots, and hence a matrix. Even though the various strings have different sizes, we'll make each matrix to be of the same size. To do that, we find the largest decimal size and the largest Roman numeral size; the vectors of all the smaller strings will be padded with zeros.

Moreover, for the Roman numeral data, we make two vectorized versions, with and without the `\t`/`\n` padding. (The version without the padding we call `target_data`.)

In [11]:
# Find the largest size for source and target strings
max_len_dec = max([len(line) for line in decimal_samples])
max_len_rn = max([len(line) for line in rn_samples])

# Make empty arrays for storing the vectorized versions of 
# the data. These are three-dimensional: the dataset is a sequence of
# strings, each string is a sequence of characters, each character is
# a one-hot vector.
tokenized_dec_samples = np.zeros((nb_samples,max_len_dec,len(decimal_chars)), dtype='float32')
tokenized_rn_samples = np.zeros((nb_samples,max_len_rn,len(rn_chars)), dtype='float32')
target_data = np.zeros((nb_samples, max_len_rn, len(rn_chars)),dtype='float32')

# For each data point
for i in range(nb_samples):
    # For each character in the decimal string for that data point...
    for k,ch in enumerate(decimal_samples[i]):
        # What is this line doing? Especially the "= 1" part?
        tokenized_dec_samples[i,k,int(ch)] = 1

    # For each character in the Roman numeral string for that data point...
    for k,ch in enumerate(rn_samples[i]):
        tokenized_rn_samples[i,k,rn_char_to_index[ch]] = 1

        # target_data will be ahead by one timestep and will not include the start character.
        if k > 0:
            target_data[i,k-1,rn_char_to_index[ch]] = 1

Let's see what these look like

In [12]:
decimal_samples[0]

'662'

In [13]:
tokenized_dec_samples[0]

array([[0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

In [14]:
rn_samples[0]

'\tDCLXII\n\n'

In [15]:
tokenized_rn_samples[0]

array([[0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

In [16]:
target_data[0]

array([[0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

### 2.3 Setting up a model

Now it's time to put together an encoder-decoder model to do this translation. There are two parameters that I want you to be able to fiddle with---how many hidden units should we have in our LSTM layer, and how many training epochs should we run? I'm putting these parameters in a separate cell to make them easier to find. However, if you re-run training with different settings to these parameters, don't forget to re-run all the cells from this point on.

In [17]:
units = 128
epochs = 50

First, the encoder model.

In [18]:
# Encoder model

# The input layer is represented by keras's Input class. The shape indicates
# that we don't know how many characters will be in a string ("None" means indefinite length)
# and that each character will be a vector of size decimal_chars
encoder_input = Input(shape=(None,len(decimal_chars)))

# The encoder proper is an LSTM layer represented by keras's LSTM class.
encoder_LSTM = LSTM(units,return_state = True) 

# That LSTM object is callable. Passing in the input layer has the effect of connection
# that input layer to the LSTM layer; as a call, that action returns a triple
# of references to the three outputs of the encoder: the output proper (which we ignore),
# the hidden state, and the context vector.
encoder_outputs, encoder_h, encoder_c = encoder_LSTM (encoder_input)

# Let's bundle the hidden state and context vector together.
encoder_states = [encoder_h, encoder_c]

Then the decoder model.

In [19]:
# Decoder model

# Its input is also of indefinite length, each character being a vector
# of length rn_chars.
decoder_input = Input(shape=(None,len(rn_chars)))

# The decoder proper is also an LSTM layer
decoder_LSTM = LSTM(units,return_sequences=True, return_state = True)

# Connect it to its input, get its output. (Its initial input is the final encoder state.)
decoder_out, _ , _ = decoder_LSTM(decoder_input, initial_state=encoder_states)

# Make a plain old neural net layer---what keras calls a "dense" layer---
# to make word predictions. This is like a "language modeling head"
# that we saw in some J&M chapters
decoder_dense = Dense(len(rn_chars),activation='softmax')
decoder_out = decoder_dense (decoder_out)

The model itself is composed of these parts, represented by the `keras` class `Model`.
See if you can make sense of the summary of the model given by the `summary()` method.

In [20]:
model = Model(inputs=[encoder_input, decoder_input],outputs=[decoder_out])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 10)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None, 9)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 128),     │     71,168 │ input_layer[0][0] │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     70,656 │ input_layer_1[0]… │
│                     │ 128), (None,      │            │ lstm[0][1],       │
│                     │ 128), (None,      │            │ lstm[0][2]        │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 9)   │      1,161 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 142,985 (558.54 KB)

 Trainable params: 142,985 (558.54 KB)

 Non-trainable params: 0 (0.00 B)

### 2.4 Training

Cue "Gonna Fly Now" from the *Rocky* movies. It's time to do some training.

The next cell compiles and fits (trains) the mode on the tokenized decimal and Roman numeral samples. The data that is passed is split into training and validation data. The validation data is like a test set, but is used for monitoring performance as the training continues rather than at the end. 

You can watch the progress as the process reports after each epoch. The report include the loss for both training and validation data; the raw numbers for the loss won't mean much, but you should observe that the loss decreases over time. If the loss isn't getting smaller, then the training is no longer improving the model.

This may take a while, especially if you have increased the number of units or epochs. You can estimate how long the training will take by looking at the time (in seconds) for the first few epochs and multiplying by the number of epochs you set it for.

In [21]:
# Run training
model.compile(optimizer='rmsprop', loss='categorical_crossentropy')
model.fit(x=[tokenized_dec_samples,tokenized_rn_samples], 
          y=target_data,
          batch_size=64,
          epochs=epochs,  # This sets the function parameter "epochs" to our variable "epochs"
          validation_split=0.2)

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 1.0502 - val_loss: 0.8418
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.8049 - val_loss: 0.7524
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7192 - val_loss: 0.7705
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6756 - val_loss: 0.6254
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6388 - val_loss: 0.6073
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6212 - val_loss: 0.5705
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.5969 - val_loss: 0.5585
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5726 - val_loss: 0.6100
Epoch 9/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5537 - val_loss: 0.5068
Epoch 10/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5381 - val_loss: 0.5397
Epoch 11/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5167 - val_loss: 0.4863
Epoch 12/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5035 - val_l

### 2.5 Testing

The way the model was set up in the earlier cells was tailored for training. Now we want to
rearrange the model so we can use it for actual translation. The following diagram from
Kompella's article explains it:

![Inference model](http://cs.wheaton.edu/~tvandrun/cs384/kompella-inference.png)


The following sets up new encoder
and decoder sides of the model. This still uses the `encodeer_input` connected to the (trained) encoder LSTM from earlier as well as the (trained) `decoder_LSTM`.

In [22]:
# Inference models for testing

# Encoder inference model
encoder_model_inf = Model(encoder_input, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(units,))
decoder_state_input_c = Input(shape=(units,))
decoder_input_states = [decoder_state_input_h, decoder_state_input_c]

decoder_out, decoder_h, decoder_c = decoder_LSTM(decoder_input, 
                                                 initial_state=decoder_input_states)

decoder_states = [decoder_h , decoder_c]

decoder_out = decoder_dense(decoder_out)

decoder_model_inf = Model(inputs=[decoder_input] + decoder_input_states,
                          outputs=[decoder_out] + decoder_states )



This function translates a single input string, which itself is a sequence of tokens. See what you can do to follow this code.

In [23]:
def decode_seq(inp_seq, print_distribution=False):
    
    # Initial states value is coming from the encoder 
    states_val = encoder_model_inf.predict(inp_seq, verbose=0)
    # We need to feed that into the decoder

    # This is the target sequence that we're going to build.
    # Its first value is the seed value of \t
    target_seq = np.zeros((1, 1, len(rn_chars)))
    target_seq[0, 0, rn_char_to_index['\t']] = 1
    
    # The translated "sentence" (actually just a Roman numeral) 
    translated_sent = ''

    # Have we gotten to the end of the translated sentence?
    stop_condition = False
    
    while not stop_condition:

        # Take one step
        decoder_out, decoder_h, decoder_c = decoder_model_inf.predict(x=[target_seq] + states_val, verbose=0)

        # decoder_out is a probability distribution over Roman numeral characters.
        if print_distribution :
            distribution = {c:decoder_out[0,-1,rn_char_to_index[c]] for c in rn_chars}
            print(str(distribution))

        # Find the most likely next character and add it to the translated sentence
        max_val_index = np.argmax(decoder_out[0,-1,:])
        sampled_rn_char = rn_index_to_char[max_val_index]
        translated_sent += sampled_rn_char

        # How do we know whether we're done? We're done if we have generated a
        # newline character as an end-of-sentence marker or if we have
        # reached the maximum length
        if ( (sampled_rn_char == '\n') or (len(translated_sent) > max_len_rn)) :
            stop_condition = True
        
        target_seq = np.zeros((1, 1, len(rn_chars)))
        target_seq[0, 0, max_val_index] = 1
        
        states_val = [decoder_h, decoder_c]
        
    return translated_sent

The `test_sentence` function allows us to pick a sample by index and test it.

In [24]:
def test_sentence(seq_index, print_distribution=False) :
    print('Input sentence:', decimal_samples[seq_index])
    inp_seq = tokenized_dec_samples[seq_index:seq_index+1]
    translated_sent = decode_seq(inp_seq, print_distribution=print_distribution)
    print('Decoded sentence:', translated_sent)

Let's try a sample. Here's what the input is and what the output should be:

In [25]:
decimal_samples[0]

'662'

In [26]:
rn_samples[0]

'\tDCLXII\n\n'

I have `print_distribution` set to `True` in this cell so that you can see the probability distribution for each output step as it progresses.

In [27]:
test_sentence(0, print_distribution=True)

Input sentence: 662
{'X': 0.0076121157, '\n': 0.0011782362, 'D': 0.86609125, 'I': 0.0038483432, 'L': 0.0309189, 'C': 0.07278236, '\t': 0.0021654305, 'M': 0.011255042, 'V': 0.0041483403}
{'X': 0.027517587, '\n': 0.0017641537, 'D': 0.056775164, 'I': 0.0027720283, 'L': 0.13640141, 'C': 0.76416576, '\t': 0.0019190295, 'M': 0.0033231168, 'V': 0.0053617638}
{'X': 0.18433918, '\n': 0.0010365599, 'D': 0.0034793743, 'I': 0.002510951, 'L': 0.7689712, 'C': 0.0338705, '\t': 0.00088293094, 'M': 0.0011433738, 'V': 0.003765877}
{'X': 0.7919597, '\n': 0.0050846776, 'D': 0.001973057, 'I': 0.13783106, 'L': 0.030151604, 'C': 0.007687885, '\t': 0.0007625755, 'M': 0.0006522405, 'V': 0.023897236}
{'X': 0.017146662, '\n': 0.011432429, 'D': 0.00084299006, 'I': 0.95292324, 'L': 0.001864471, 'C': 0.0017160069, '\t': 0.00044251294, 'M': 0.0002906635, 'V': 0.013341004}
{'X': 0.07048637, '\n': 0.17834744, 'D': 0.0070952773, 'I': 0.6402187, 'L': 0.014124861, 'C': 0.015442747, '\t': 0.010821647, 'M': 0.008561411, 'V

How does it do over all? Let's observe its performance on 10 (or however many you choose) samples. This will print the output Roman numeral together with the correct answer in parentheses and an indication whether the model got it right.

In [28]:
for seq_index in range(10):
    inp_seq = tokenized_dec_samples[seq_index:seq_index+1]
    translated_sent = decode_seq(inp_seq)
    print('-')
    print('Input sentence: ' +  decimal_samples[seq_index])
    print('Decoded sentence: ' + translated_sent.strip() + ' (' + rn_samples[seq_index].strip() + ') ' + 
          str(translated_sent.strip() == rn_samples[seq_index].strip()))

-
Input sentence: 662
Decoded sentence: DCLXII (DCLXII) True
-
Input sentence: 3775
Decoded sentence: MMMDCCLXV (MMMDCCLXXV) False
-
Input sentence: 1813
Decoded sentence: MDCCCIII (MDCCCXIII) False
-
Input sentence: 545
Decoded sentence: DLXV (DXLV) False
-
Input sentence: 2632
Decoded sentence: MMDCXXII (MMDCXXXII) False
-
Input sentence: 3453
Decoded sentence: MMMCDLII (MMMCDLIII) False
-
Input sentence: 1647
Decoded sentence: MDCXVII (MDCXLVII) False
-
Input sentence: 3820
Decoded sentence: MMMDCCCX (MMMDCCCXX) False
-
Input sentence: 2894
Decoded sentence: MMDCCCXII (MMDCCCXCIV) False
-
Input sentence: 121
Decoded sentence: CXXI (CXXI) True


The following tests the accuracy over 250 (or however many you choose) samples. This may take a few minutes.

In [29]:
print(np.mean([decode_seq(tokenized_dec_samples[i:i+1]).strip() == rn_samples[i].strip() for i in range(250)]))

0.188


To make the model perform better, go back and adjust the number of epochs (and/or, possibly, the number of units).

## 3. The Canturbery Tales

I figured a good problem to try this on would be translating Chaucer's *Canterbury Tales* from old English to modern English, since the languages aren't so different and because a parallel text is readily available. (I got this from [Harvard University's Geoffrey Chaucer Website](https://chaucer.fas.harvard.edu/pages/text-and-translations).

Take a look at `/homes/tvandrun/Public/cs384/canterbury.txt`, which contains the prologue and the Knight's Tale, a little more than 3000 lines in total.

The rest of this lab proceeds without commentary, but the code is very similar to the code above (one modification is that the set of characters is determined by the characters found in the text itself instead of determined ahead of time). You should follow the code as best you can, and take note of things to modify and try.

In [30]:
lines = open('/homes/tvandrun/Public/cs384/canterbury.txt', encoding='utf-8').read().split('\n')

In [31]:
old_eng_sent = []
new_eng_sent = []
old_eng_chars = set()
new_eng_chars = set()
nb_samples = 3000

for line in range(nb_samples):
    
    old_eng_line = str(lines[line]).split('\t')[0]
    
    # Append '\t' for start of the sentence and '\n' to signify end of the sentence
    new_eng_line = '\t' + str(lines[line]).split('\t')[1].strip() + '\n\n\n'
    old_eng_sent.append(old_eng_line)
    new_eng_sent.append(new_eng_line)
    
    for ch in old_eng_line:
        if (ch not in old_eng_chars):
            old_eng_chars.add(ch)
            
    for ch in new_eng_line:
        if (ch not in new_eng_chars):
            new_eng_chars.add(ch)

In [32]:
old_eng_sent[0]

'Whan that Aprill with his shoures soote'

In [33]:
new_eng_sent[0]

'\tWhen April with its sweet-smelling showers\n\n\n'

In [34]:
new_eng_chars = sorted(list(new_eng_chars))
old_eng_chars = sorted(list(old_eng_chars))

In [35]:
# dictionary to index each english character - key is index and value is english character
old_eng_index_to_char_dict = {}

# dictionary to get english character given its index - key is english character and value is index
old_eng_char_to_index_dict = {}

for k, v in enumerate(old_eng_chars):
    old_eng_index_to_char_dict[k] = v
    old_eng_char_to_index_dict[v] = k

In [36]:
# dictionary to index each modern English character - key is index and value is character
new_eng_index_to_char_dict = {}

# dictionary to get modern English character given its index - key is french character and value is index
new_eng_char_to_index_dict = {}
for k, v in enumerate(new_eng_chars):
    new_eng_index_to_char_dict[k] = v
    new_eng_char_to_index_dict[v] = k

In [37]:
max_len_old_eng_sent = max([len(line) for line in old_eng_sent])
max_len_new_eng_sent = max([len(line) for line in new_eng_sent])

In [44]:
tokenized_old_eng_sentences = np.zeros(shape = (nb_samples,max_len_old_eng_sent,len(old_eng_chars)), dtype='float32')
tokenized_new_eng_sentences = np.zeros(shape = (nb_samples,max_len_new_eng_sent,len(new_eng_chars)), dtype='float32')
target_data = np.zeros((nb_samples, max_len_new_eng_sent, len(new_eng_chars)),dtype='float32')

In [45]:
# Vectorize 

for i in range(nb_samples):
    for k,ch in enumerate(old_eng_sent[i]):
        tokenized_old_eng_sentences[i,k,old_eng_char_to_index_dict[ch]] = 1
        
    for k,ch in enumerate(new_eng_sent[i]):
        tokenized_new_eng_sentences[i,k,new_eng_char_to_index_dict[ch]] = 1

        # decoder_target_data will be ahead by one timestep and will not include the start character.
        if k > 0:
            target_data[i,k-1,new_eng_char_to_index_dict[ch]] = 1

In [46]:
# Change this to increase the power
units = 256
epochs = 100

In [47]:
# Encoder model

encoder_input = Input(shape=(None,len(old_eng_chars)))
encoder_LSTM = LSTM(units,return_state = True) 
encoder_outputs, encoder_h, encoder_c = encoder_LSTM (encoder_input)
encoder_states = [encoder_h, encoder_c]

In [48]:
# Decoder model

decoder_input = Input(shape=(None,len(new_eng_chars)))
decoder_LSTM = LSTM(units,return_sequences=True, return_state = True)
decoder_out, _ , _ = decoder_LSTM(decoder_input, initial_state=encoder_states)
decoder_dense = Dense(len(new_eng_chars),activation='softmax')
decoder_out = decoder_dense (decoder_out)

In [49]:
model = Model(inputs=[encoder_input, decoder_input],outputs=[decoder_out])

# Run training
model.compile(optimizer='rmsprop', loss='categorical_crossentropy')
model.fit(x=[tokenized_old_eng_sentences,tokenized_new_eng_sentences], 
          y=target_data,
          batch_size=64,
          epochs=epochs, 
          validation_split=0.2)

Epoch 1/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 136ms/step - loss: 2.0338 - val_loss: 1.7453
Epoch 2/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 132ms/step - loss: 1.7275 - val_loss: 1.6705
Epoch 3/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 134ms/step - loss: 1.6997 - val_loss: 1.6586
Epoch 4/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 133ms/step - loss: 1.6811 - val_loss: 1.6748
Epoch 5/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 133ms/step - loss: 1.6678 - val_loss: 1.6837
Epoch 6/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 131ms/step - loss: 1.6708 - val_loss: 1.6301
Epoch 7/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 134ms/step - loss: 1.6550 - val_loss: 1.6506
Epoch 8/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 134ms/step - loss: 1.6531 - val_loss: 1.6181
Epoch 9/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 132ms/step - loss: 1.6326 - val_loss: 1.6010
Epoch 10/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 132ms/step - loss: 1.6166 - val_loss: 1.5814
Epoch 11/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 133ms/step - loss: 1.6181 - val_loss: 1.5675
Epoch 12/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 5

In [50]:
# Inference models for testing

# Encoder inference model
encoder_model_inf = Model(encoder_input, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(units,))
decoder_state_input_c = Input(shape=(units,))
decoder_input_states = [decoder_state_input_h, decoder_state_input_c]

decoder_out, decoder_h, decoder_c = decoder_LSTM(decoder_input, 
                                                 initial_state=decoder_input_states)

decoder_states = [decoder_h , decoder_c]

decoder_out = decoder_dense(decoder_out)

decoder_model_inf = Model(inputs=[decoder_input] + decoder_input_states,
                          outputs=[decoder_out] + decoder_states )

In [51]:
def decode_seq(inp_seq):
    
    # Initial states value is coming from the encoder 
    states_val = encoder_model_inf.predict(inp_seq, verbose=0)
    
    target_seq = np.zeros((1, 1, len(new_eng_chars)))
    target_seq[0, 0, new_eng_char_to_index_dict['\t']] = 1
    
    translated_sent = ''
    stop_condition = False
    
    while not stop_condition:
        
        decoder_out, decoder_h, decoder_c = decoder_model_inf.predict(x=[target_seq] + states_val, verbose=0)
        
        max_val_index = np.argmax(decoder_out[0,-1,:])
        sampled_new_eng_char = new_eng_index_to_char_dict[max_val_index]
        translated_sent += sampled_new_eng_char
        
        if ( (sampled_new_eng_char == '\n') or (len(translated_sent) > max_len_new_eng_sent)) :
            stop_condition = True
        
        target_seq = np.zeros((1, 1, len(new_eng_chars)))
        target_seq[0, 0, max_val_index] = 1
        
        states_val = [decoder_h, decoder_c]
        
    return translated_sent

In [52]:
for seq_index in range(10):
    inp_seq = tokenized_old_eng_sentences[seq_index:seq_index+1]
    translated_sent = decode_seq(inp_seq)
    print('-')
    print('Input sentence:', old_eng_sent[seq_index])
    print('Decoded sentence:', translated_sent)

-
Input sentence: Whan that Aprill with his shoures soote
Decoded sentence: The was and the was and the wasthe the ware,

-
Input sentence: The droghte of March hath perced to the roote,
Decoded sentence: The was and the was and the wasthe the ware,

-
Input sentence: And bathed every veyne in swich licour
Decoded sentence: And the was and the was and the ware the ware,

-
Input sentence: Of which vertu engendred is the flour;
Decoded sentence: And the was and the was and the ware the ware,

-
Input sentence: Whan Zephirus eek with his sweete breeth
Decoded sentence: The was and the was and the wasthe the ware,

-
Input sentence: Inspired hath in every holt and heeth
Decoded sentence: And the was and the was and the ware the ware,

-
Input sentence: The tendre croppes, and the yonge sonne
Decoded sentence: The was and the was and the wasthe the ware,

-
Input sentence: Hath in the Ram his half cours yronne,
Decoded sentence: The was and the was and the wasthe the ware,

-
Input sentenc